## Ingesting Data with `SQL` and `CTAS (Create Table As Select)`

### Creating Table from Raw Files in UC Volume with CTAS

In [ ]:
%sql

CREATE TABLE IF NOT EXISTS ev_lab.bronze.ev_station_events
USING DELTA
AS SELECT
    CAST(event_id AS STRING) AS event_id,
    CAST(station_id AS STRING) AS station_id,
    CAST(charger_id AS STRING) AS charger_id,
    CAST(event_time AS TIMESTAMP) AS event_time,
    CAST(event_type AS STRING) AS event_type,
    CAST(severity AS STRING) AS severity,
    CAST(description AS STRING) AS description,
    CAST(resolved AS BOOLEAN) AS resolved
FROM read_files(
    '/Volumes/ev_lab/bronze/raw_files/EV_station_events.csv',
    format => 'csv',
    header => true
);

In [ ]:
%sql

SELECT * FROM ev_lab.bronze.ev_station_events

### Generate Silver Layer Summary with CTAS

In [ ]:
%sql

CREATE OR REPLACE TABLE ev_lab.silver.ev_charging_sessions_summary
PARTITIONED BY (station_id)
AS
SELECT
    session_id,
    station_id,
    charger_id,

    CAST(start_time AS TIMESTAMP) AS start_time,
    CAST(end_time AS TIMESTAMP) AS end_time,

    CAST(energy_kwh AS DOUBLE) AS energy_kwh,
    CAST(charging_rate_kw AS DOUBLE) AS charging_rate_kw,
    CAST(temperature_c AS DOUBLE) AS temperature_c,

    status,

    -- Key metrics
    (UNIX_TIMESTAMP(end_time) - UNIX_TIMESTAMP(start_time)) / 3600 AS duration_hours,

    CASE 
        WHEN charging_rate_kw > 0 THEN energy_kwh / charging_rate_kw
        ELSE NULL
    END AS efficiency,

    -- Time features
    DATE(start_time) AS session_date,
    HOUR(start_time) AS session_hour,

    -- Simple flags
    CASE WHEN status != 'Completed' THEN true ELSE false END AS is_issue,
    CASE WHEN energy_kwh = 0 THEN true ELSE false END AS is_fault

FROM ev_lab.bronze.ev_charging_sessions
WHERE session_id IS NOT NULL;

In [ ]:
%sql

SELECT * FROM ev_lab.silver.ev_charging_sessions_summary